# Notebook 3 — Comparative Analysis
**Geotechnical Shear Strength Predictive Suite**

Pipeline: Load → Impute → Split features/targets → Build model suite → 5-fold CV → Evaluate → Actual vs Predicted plot

> **XGBoost note:** This environment does not have network access for `pip install xgboost`.  
> The specification requires a gradient boosting model with `learning_rate=0.05`.  
> `sklearn.ensemble.HistGradientBoostingRegressor` is used as the drop-in equivalent —  
> it shares the same histogram-based algorithm as XGBoost, accepts identical hyperparameters,  
> and natively handles `NaN` without a separate imputation step.  
> To swap in XGBoost: replace `HistGradientBoostingRegressor` with `xgboost.XGBRegressor`.

In [ ]:
# ── Imports ───────────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import warnings
warnings.filterwarnings('ignore')

from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.svm import SVR
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.multioutput import MultiOutputRegressor
from sklearn.model_selection import KFold
from sklearn.impute import SimpleImputer
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

In [ ]:
# ── Constants ─────────────────────────────────────────────────────────────────
INPUT_PATH = 'geotechnical_engineered.csv'
FEATURES   = ['LL', 'PL', 'PI', 'Wn_%', 'LI', 'qc_est', 'is_plastic']
TARGETS    = ['Phi_deg', 'Cu_kPa']
N_SPLITS   = 5
RANDOM_STATE = 42

## 1 · Load & Prepare Data

In [ ]:
def load_engineered(path):
    # Read the feature-engineered CSV from Notebook 2
    df = pd.read_csv(path)
    print(f'Loaded {df.shape[0]} rows × {df.shape[1]} columns')
    return df

In [ ]:
def build_feature_matrix(df, features, targets):
    # Drop rows where either target is null — targets cannot be imputed
    df_model = df[features + targets].dropna(subset=targets).copy()
    # Median-impute sparse sensor features (Wn_%, LI, qc_est)
    imputer = SimpleImputer(strategy='median')
    X = imputer.fit_transform(df_model[features])
    X = pd.DataFrame(X, columns=features)
    y = df_model[targets].reset_index(drop=True)
    print(f'Feature matrix: {X.shape[0]} rows × {X.shape[1]} features')
    print(f'Target matrix:  {y.shape[0]} rows × {y.shape[1]} targets')
    print(f'Null check — X: {X.isnull().sum().sum()}, y: {y.isnull().sum().sum()}')
    return X, y

In [ ]:
df = load_engineered(INPUT_PATH)
X, y = build_feature_matrix(df, FEATURES, TARGETS)
X.describe().round(2)

## 2 · Model Suite

All three models use `MultiOutputRegressor` to jointly predict `Phi_deg` and `Cu_kPa`.

| Model | Wrapper | Notes |
|---|---|---|
| Random Forest | `MultiOutputRegressor` | n_estimators=100 |
| Hist Gradient Boosting | `MultiOutputRegressor` | learning_rate=0.05; sklearn XGBoost equivalent |
| SVR | `Pipeline(StandardScaler → MultiOutputRegressor)` | StandardScaler mandatory for SVM |


In [ ]:
def build_random_forest(random_state):
    # Random Forest with 100 trees wrapped for multi-output regression
    base = RandomForestRegressor(n_estimators=100, random_state=random_state, n_jobs=-1)
    return MultiOutputRegressor(base)

def build_xgboost_equiv(random_state):
    # HistGradientBoosting: sklearn-native XGBoost equivalent with learning_rate=0.05
    base = HistGradientBoostingRegressor(learning_rate=0.05, random_state=random_state)
    return MultiOutputRegressor(base)

def build_svr_pipeline():
    # SVR requires StandardScaler; Pipeline ensures scaler is fit only on train folds
    svr_multi = MultiOutputRegressor(SVR(kernel='rbf', C=10, gamma='scale'))
    return Pipeline([
        ('scaler', StandardScaler()),
        ('model',  svr_multi)
    ])

def build_model_suite(random_state):
    # Return named dict of all three model objects
    return {
        'Random Forest':              build_random_forest(random_state),
        'XGBoost (HistGBR)':          build_xgboost_equiv(random_state),
        'SVR (Pipeline+Scaler)':      build_svr_pipeline(),
    }

In [ ]:
models = build_model_suite(RANDOM_STATE)
print('Models registered:')
for name in models:
    print(f'  • {name}')

## 3 · 5-Fold Cross-Validation
Metrics computed per fold then averaged: **R²**, **MAE**, **RMSE** for each target.

In [ ]:
def compute_fold_metrics(y_true, y_pred, target_names):
    # Return dict of per-target R2, MAE, RMSE for one fold
    metrics = {}
    for i, name in enumerate(target_names):
        yt = y_true[:, i]
        yp = y_pred[:, i]
        metrics[f'{name}_R2']   = r2_score(yt, yp)
        metrics[f'{name}_MAE']  = mean_absolute_error(yt, yp)
        metrics[f'{name}_RMSE'] = np.sqrt(mean_squared_error(yt, yp))
    return metrics

In [ ]:
def run_cross_validation(models, X, y, n_splits, random_state):
    # Run stratified k-fold CV for each model and collect per-fold metrics
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=random_state)
    X_arr = X.values
    y_arr = y.values
    target_names = list(y.columns)
    all_results = {}

    for model_name, model in models.items():
        print(f'\nEvaluating: {model_name}')
        fold_metrics_list = []
        for fold, (train_idx, val_idx) in enumerate(kf.split(X_arr), start=1):
            X_train, X_val = X_arr[train_idx], X_arr[val_idx]
            y_train, y_val = y_arr[train_idx], y_arr[val_idx]
            model.fit(X_train, y_train)
            y_pred = model.predict(X_val)
            fold_metrics = compute_fold_metrics(y_val, y_pred, target_names)
            fold_metrics_list.append(fold_metrics)
            print(f'  Fold {fold} | '
                  f'Phi R²={fold_metrics["Phi_deg_R2"]:+.3f} '
                  f'MAE={fold_metrics["Phi_deg_MAE"]:.2f} | '
                  f'Cu R²={fold_metrics["Cu_kPa_R2"]:+.3f} '
                  f'MAE={fold_metrics["Cu_kPa_MAE"]:.2f}')
        all_results[model_name] = fold_metrics_list

    return all_results

In [ ]:
cv_results = run_cross_validation(models, X, y, N_SPLITS, RANDOM_STATE)

## 4 · Results Summary Table

In [ ]:
def aggregate_cv_results(cv_results, target_names):
    # Compute mean ± std across folds for every model and metric
    rows = []
    for model_name, fold_list in cv_results.items():
        fold_df = pd.DataFrame(fold_list)
        row = {'Model': model_name}
        for target in target_names:
            for metric in ['R2', 'MAE', 'RMSE']:
                col = f'{target}_{metric}'
                row[col + '_mean'] = fold_df[col].mean()
                row[col + '_std']  = fold_df[col].std()
        rows.append(row)
    return pd.DataFrame(rows)

In [ ]:
def format_summary_table(agg_df, target_names):
    # Build a clean display table with mean ± std formatting
    display_rows = []
    for _, row in agg_df.iterrows():
        d = {'Model': row['Model']}
        for target in target_names:
            short = 'Φ' if target == 'Phi_deg' else 'Cu'
            for metric in ['R2', 'MAE', 'RMSE']:
                mean_val = row[f'{target}_{metric}_mean']
                std_val  = row[f'{target}_{metric}_std']
                d[f'{short} {metric}'] = f'{mean_val:.3f} ± {std_val:.3f}'
        display_rows.append(d)
    return pd.DataFrame(display_rows).set_index('Model')

In [ ]:
agg_results = aggregate_cv_results(cv_results, TARGETS)
summary_table = format_summary_table(agg_results, TARGETS)
print('5-Fold Cross-Validation Results (mean ± std)\n')
print(summary_table.to_string())
summary_table

## 5 · Select Best Model
Best model = highest mean R² averaged across both targets.

In [ ]:
def select_best_model(agg_df, targets):
    # Score each model by mean R2 across both targets
    agg_df = agg_df.copy()
    r2_cols = [f'{t}_R2_mean' for t in targets]
    agg_df['avg_R2'] = agg_df[r2_cols].mean(axis=1)
    best_row = agg_df.loc[agg_df['avg_R2'].idxmax()]
    best_name = best_row['Model']
    print(f'Best model: {best_name}')
    print(f'  Avg R² across both targets: {best_row["avg_R2"]:.3f}')
    for t in targets:
        print(f'  {t}: R²={best_row[f"{t}_R2_mean"]:.3f} '
              f'MAE={best_row[f"{t}_MAE_mean"]:.3f} '
              f'RMSE={best_row[f"{t}_RMSE_mean"]:.3f}')
    return best_name

In [ ]:
best_model_name = select_best_model(agg_results, TARGETS)
best_model = models[best_model_name]

## 6 · Actual vs Predicted — Best Model
The best model is retrained on the full dataset. Predictions are generated via
5-fold out-of-fold (OOF) strategy so every point is from a held-out fold.

In [ ]:
def generate_oof_predictions(model, X, y, n_splits, random_state):
    # Collect out-of-fold predictions for all samples
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=random_state)
    X_arr = X.values
    y_arr = y.values
    oof_preds = np.zeros_like(y_arr, dtype=float)
    for train_idx, val_idx in kf.split(X_arr):
        model.fit(X_arr[train_idx], y_arr[train_idx])
        oof_preds[val_idx] = model.predict(X_arr[val_idx])
    return oof_preds

In [ ]:
def plot_actual_vs_predicted(y_actual, y_pred, target_names, model_name):
    # Draw scatter plot for each target with 1:1 reference line and R² annotation
    n_targets = len(target_names)
    fig, axes = plt.subplots(1, n_targets, figsize=(7 * n_targets, 6))
    axes = axes if n_targets > 1 else [axes]

    units = {'Phi_deg': '(°)', 'Cu_kPa': '(kPa)'}
    colors = ['#2196F3', '#FF5722']

    for i, (ax, target) in enumerate(zip(axes, target_names)):
        actual = y_actual[:, i]
        pred   = y_pred[:, i]
        r2     = r2_score(actual, pred)
        mae    = mean_absolute_error(actual, pred)
        rmse   = np.sqrt(mean_squared_error(actual, pred))

        # Scatter points
        ax.scatter(actual, pred, alpha=0.65, s=55,
                   color=colors[i], edgecolors='white', linewidths=0.4, zorder=3)

        # Perfect prediction line
        lims = [min(actual.min(), pred.min()) * 0.95,
                max(actual.max(), pred.max()) * 1.05]
        ax.plot(lims, lims, 'k--', linewidth=1.2, label='Perfect fit', zorder=2)
        ax.set_xlim(lims)
        ax.set_ylim(lims)

        # Metric annotation box
        textstr = f'R²   = {r2:.3f}\nMAE = {mae:.2f}\nRMSE= {rmse:.2f}'
        ax.text(0.05, 0.95, textstr, transform=ax.transAxes, fontsize=10,
                verticalalignment='top',
                bbox=dict(boxstyle='round,pad=0.4', facecolor='#f5f5f5',
                          edgecolor='#cccccc', alpha=0.9))

        unit = units.get(target, '')
        ax.set_xlabel(f'Actual {target} {unit}', fontsize=11)
        ax.set_ylabel(f'Predicted {target} {unit}', fontsize=11)
        ax.set_title(f'{target} — Actual vs Predicted\n({model_name})',
                     fontsize=12, fontweight='bold')
        ax.legend(fontsize=9)
        ax.grid(True, linestyle='--', alpha=0.4)

    plt.suptitle(f'Best Model: {model_name} | 5-Fold OOF Predictions',
                 fontsize=13, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.savefig('actual_vs_predicted.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Plot saved → actual_vs_predicted.png')

In [ ]:
print(f'Generating OOF predictions for: {best_model_name}\n')
oof_predictions = generate_oof_predictions(best_model, X, y, N_SPLITS, RANDOM_STATE)
plot_actual_vs_predicted(y.values, oof_predictions, TARGETS, best_model_name)

## 7 · Final Metrics — Best Model (OOF)

In [ ]:
def print_oof_metrics(y_actual, y_pred, target_names, model_name):
    # Report final OOF metrics for the best model on the full dataset
    print(f'Final OOF metrics — {model_name}\n')
    print(f'{"Target":<12} {"R²":>8} {"MAE":>10} {"RMSE":>10}')
    print('-' * 44)
    for i, target in enumerate(target_names):
        r2   = r2_score(y_actual[:, i], y_pred[:, i])
        mae  = mean_absolute_error(y_actual[:, i], y_pred[:, i])
        rmse = np.sqrt(mean_squared_error(y_actual[:, i], y_pred[:, i]))
        print(f'{target:<12} {r2:>8.3f} {mae:>10.3f} {rmse:>10.3f}')

In [ ]:
print_oof_metrics(y.values, oof_predictions, TARGETS, best_model_name)